# Study 816 — Drawdown Duration — the teardown

The per-leg splits, the Newey-West spread *t*, the pooled Welch book test, the 1,000-permutation two-sided placebo, the two-era robustness cut, the costed timer, and the 20-seed synthetic control.

In [1]:
R = {'start': '2010-01-04', 'end': '2026-06-30', 'n_names': 50, 'n_rows': 4147, 'n_days': 3895, 'fingerprint': '357fd262912f', 'spread_bps': -1.12, 't_nw': -0.8, 't_1s': -0.81, 'hi_bps': 6.62, 'lo_bps': 7.75, 'welch_t': -0.41, 'gross_sharpe': -0.2, 'placebo_obs': -1.12, 'placebo_mean': -0.037, 'placebo_sd': 1.007, 'placebo_p': 0.261, 'placebo_sigma': 1.12, 'placebo_draws': 1000, 'era_early_bps': 0.5, 'era_early_t': 0.28, 'era_early_n': 1761, 'era_late_bps': -2.46, 'era_late_t': -1.17, 'era_late_n': 2134, 'timer_1_gross': -1.12, 'timer_1_cost': 2.14, 'timer_1_net': -3.26, 'timer_1_t': -2.34, 'timer_5_gross': -1.12, 'timer_5_cost': 10.14, 'timer_5_net': -11.26, 'timer_5_t': -8.07, 'null_mean_t': 0.1, 'null_sd_t': 1.15, 'null_fire': 2, 'planted_t': -15.31, 'planted_welch': -14.44}

## The headline — long-high-underwater / short-low-underwater spread

Daily equal-weight top-30% (high time-underwater) minus bottom-30% (low) spread, n = 3,895 days, as-of 2026-06-30, fingerprint `357fd262912f`.

In [2]:
print(f"spread        : {R['spread_bps']:+.2f} bps/day  NW(10) t = {R['t_nw']:+.2f}  "
      f"one-sample t = {R['t_1s']:+.2f}")
print(f"books         : high-UW {R['hi_bps']:+.2f} vs low-UW {R['lo_bps']:+.2f} bps "
      f"(Welch t = {R['welch_t']:+.2f})")
print(f"gross Sharpe  : {R['gross_sharpe']:.2f} (before cost)")

spread        : -1.12 bps/day  NW(10) t = -0.80  one-sample t = -0.81
books         : high-UW +6.62 vs low-UW +7.75 bps (Welch t = -0.41)
gross Sharpe  : -0.20 (before cost)


## Placebo — column-permute the forward returns (1,000 permutations, two-sided)

In [3]:
print(f"observed {R['placebo_obs']:+.2f} bps vs placebo mean {R['placebo_mean']:+.3f} "
      f"(sd {R['placebo_sd']:.3f}) -> {R['placebo_sigma']:.2f} sigma, two-sided p = {R['placebo_p']:.3f}")

observed -1.12 bps vs placebo mean -0.037 (sd 1.007) -> 1.12 sigma, two-sided p = 0.261


## Robustness — two eras (split 2018-01-01)

In [4]:
print(f"2010-2017 (n={R['era_early_n']}): {R['era_early_bps']:+.2f} bps  NW t = {R['era_early_t']:+.2f}")
print(f"2018-2026 (n={R['era_late_n']}): {R['era_late_bps']:+.2f} bps  NW t = {R['era_late_t']:+.2f}")
print('sign flips across eras and neither half is significant -> a textbook null')

2010-2017 (n=1761): +0.50 bps  NW t = +0.28
2018-2026 (n=2134): -2.46 bps  NW t = -1.17
sign flips across eras and neither half is significant -> a textbook null


## The timer — can you get paid for it?

2 sides × one-way cost × NAV per day on the long-short book; short pays 50 bps/yr borrow.

In [5]:
for tag,g,c,n,t in [('1 bp',R['timer_1_gross'],R['timer_1_cost'],R['timer_1_net'],R['timer_1_t']),
                    ('5 bps',R['timer_5_gross'],R['timer_5_cost'],R['timer_5_net'],R['timer_5_t'])]:
    print(f"{tag:>5} one-way: gross {g:+.2f} -> net {n:+.2f} bps/day (cost {c:.2f}/day, t={t:+.2f})")

 1 bp one-way: gross -1.12 -> net -3.26 bps/day (cost 2.14/day, t=-2.34)
5 bps one-way: gross -1.12 -> net -11.26 bps/day (cost 10.14/day, t=-8.07)


## Synthetic positive control — the machinery is unbiased

Live: the detector must NOT fire on the null and must recover a planted relation (low-drift names stay underwater and keep sinking -> a *negative* high-minus-low spread).

In [6]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath(os.path.join('..','..','..')))
import numpy as np
from drawdown_duration import data, strategy as st
null_t = np.array([st.synthetic_detect(data.synthetic_panel(knob=0.0, seed=816+s, n_assets=40, n_days=1200))['t_nw'] for s in range(8)])
print(f"null (knob=0), 8 seeds: NW t mean {null_t.mean():+.2f} (sd {null_t.std(ddof=1):.2f}), |t|>=2 in {(abs(null_t)>=2).sum()}/8")
planted = st.synthetic_detect(data.synthetic_panel(knob=0.0010, seed=816, n_assets=40, n_days=1500))
print(f"planted (knob=0.0010): NW t = {planted['t_nw']:+.2f}, Welch t = {planted['welch_t']:+.2f}")

null (knob=0), 8 seeds: NW t mean -0.31 (sd 1.35), |t|>=2 in 1/8


planted (knob=0.0010): NW t = -15.31, Welch t = -14.44


## Verdict

- **Signal — None.** The fraction of the trailing year a mega-cap spent underwater does **not** predict its forward return: the long-high-underwater / short-low-underwater spread is **-1.12 bps/day** (NW *t* = **-0.80**), ≈1.12σ from a 1,000-permutation null (two-sided p = 0.26), with a sign that *flips* between eras (*t* = +0.28 / -1.17). The 20-seed synthetic control recovers a *planted* relation cleanly (*t* = -15.31, fires on 2/20 nulls), so the flat real-tape result is a genuine absence of signal. Survivorship understates any 'losers keep sinking' tilt.
- **Tradability — Mirage.** The book loses gross (-1.12 bps/day) and net (**-3.26 bps/day** at 1 bp, -11.26 at 5 bps); the 2.14 bps/day round-trip friction eats even the sign-flip at a mere 1 bp.